In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kritikseth/fruit-and-vegetable-image-recognition")

print("Path to dataset files:", path)


Path to dataset files: /home/greg/.cache/kagglehub/datasets/kritikseth/fruit-and-vegetable-image-recognition/versions/8


In [ ]:
# Import libraries
import os
import cv2 as cv
import torch
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt


GPU Available: False


In [4]:

dataset_path = path
train_dir = os.path.join(dataset_path, 'train')
test_dir = os.path.join(dataset_path, 'test')
model = YOLO("yolo11n-cls.pt")

In [35]:


# train the model
results = model.train(
    data=train_dir, 
    epochs=20, 
    imgsz=224,
    hsv_s=0.7,   # Randomly vary saturation by 70%
    hsv_v=0.4,   # Randomly vary brightness by 40%
    degrees=15,  
    flipud=0.5   
)

Ultralytics 8.4.30 🚀 Python-3.13.9 torch-2.7.0 CPU (Intel Core i7-10700KF 3.80GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/greg/.cache/kagglehub/datasets/kritikseth/fruit-and-vegetable-image-recognition/versions/8/train, degrees=15, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/classify/fruit_veg_yolo11n2/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train

In [38]:
def predict_frame(model, frame):
    h, w, _ = frame.shape
    size = min(h, w) # Get the smallest dimension to make a square
    
    # Calculate coordinates for a center crop
    start_x = (w - size) // 2
    start_y = (h - size) // 2
    
    # Crop the image to just the center square
    cropped_frame = frame[start_y:start_y+size, start_x:start_x+size]
    
    # Optional: Draw a box on the original frame so you know where to hold the fruit
    cv.rectangle(frame, (start_x, start_y), (start_x+size, start_y+size), (255, 0, 0), 2)

    results = model.predict(cropped_frame, verbose=False)
    
    if results and len(results) > 0:
        probs = results[0].probs
        return results[0].names[probs.top1], probs.top1conf.item()
    return "None", 0.0

def boost_vibrancy(frame):
    # 1. Convert to HSV (Hue, Saturation, Value)
    hsv = cv.cvtColor(frame, cv.COLOR_BGR2HSV).astype("float32")
    
    # 2. Boost Saturation (the middle channel) by 1.5x
    hsv[:, :, 1] = hsv[:, :, 1] * 1.3
    hsv[:, :, 1] = np.clip(hsv[:, :, 1], 0, 255) # Keep in 0-255 range
    
    # 3. Convert back to BGR
    vibrant_frame = cv.cvtColor(hsv.astype("uint8"), cv.COLOR_HSV2BGR)
    
    # 4. Light Contrast Boost (Alpha > 1.0)
    vibrant_frame = cv.convertScaleAbs(vibrant_frame, alpha=1.2, beta=10)
    
    return vibrant_frame

In [40]:
##
# Opens a video capture device with a resolution of 800x600
# at 30 FPS.
##
def open_camera(cam_id = 0):
    # Specify the backend (CAP_V4L2) for Linux
    cap = cv.VideoCapture(cam_id, cv.CAP_V4L2)
    
    # Check if it actually opened
    if not cap.isOpened():
        print(f"Could not open camera {cam_id}. Trying index 1...")
        cap = cv.VideoCapture(1, cv.CAP_V4L2) # Sometimes the ID shifts to 1
        
    cap.set(cv.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv.CAP_PROP_FRAME_HEIGHT, 480)
    return cap
 
##
# Gets a frame from an open video device, or returns None
# if the capture could not be made.
##
def get_frame(device):
    ret, img = device.read()
    if (ret == False): # failed to capture
        print("Failed to capture from camera. Check if it is connected and try again.")
        return None
    return img
 
##
# Closes all OpenCV windows and releases video capture device
# before exit.
##
def cleanup(device): 
    cv.destroyAllWindows()
    device.release()
 

########### Main Program ###########
if __name__ == "__main__":
    # 1. Load the model here!
    model = YOLO("runs/classify/train/weights/best.pt")
    
    camera_id = 0
    dev = open_camera(camera_id)

    while True:
        img_orig = get_frame(dev)
        if img_orig is None:
            break
        
        # Get prediction
        img_vibrant = boost_vibrancy(img_orig)
        pred_class, confidence = predict_frame(model, img_vibrant)

        # 2. Only display if confidence is high (e.g., > 50%)
        if confidence > 0.50:
            label = f"{pred_class} ({confidence*100:.1f}%)"
            # Draw the text on the image
            cv.putText(img_vibrant, label, (50, 50), cv.FONT_HERSHEY_SIMPLEX, 
                       1, (0, 255, 0), 2, cv.LINE_AA)
        
        cv.imshow("Fruit & Veg Detection", img_vibrant)

        if cv.waitKey(1) == 27: # ESC to quit
            break
 
    cleanup(dev)

[ WARN:0@5459.377] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ WARN:0@5459.377] global cap.cpp:478 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by index


Could not open camera 0. Trying index 1...


QObject::moveToThread: Current thread (0x2c417b40) is not the object's thread (0x35afdee0).
Cannot move to target thread (0x2c417b40)

QObject::moveToThread: Current thread (0x2c417b40) is not the object's thread (0x35afdee0).
Cannot move to target thread (0x2c417b40)

QObject::moveToThread: Current thread (0x2c417b40) is not the object's thread (0x35afdee0).
Cannot move to target thread (0x2c417b40)

QObject::moveToThread: Current thread (0x2c417b40) is not the object's thread (0x35afdee0).
Cannot move to target thread (0x2c417b40)

QObject::moveToThread: Current thread (0x2c417b40) is not the object's thread (0x35afdee0).
Cannot move to target thread (0x2c417b40)

QObject::moveToThread: Current thread (0x2c417b40) is not the object's thread (0x35afdee0).
Cannot move to target thread (0x2c417b40)

QObject::moveToThread: Current thread (0x2c417b40) is not the object's thread (0x35afdee0).
Cannot move to target thread (0x2c417b40)

QObject::moveToThread: Current thread (0x2c417b40) is n